## Final Fix 

In [ ]:
import pandas as pd
import numpy as np

# ------------------------------------------------------------------
# 1. Load the raw file
# ------------------------------------------------------------------
file_path = r"D:\ITC\Intern Project\Data Analysis Project\Summary Data Automation\20260531-Income and Expensed-Jan to July 26.xlsx"
df_raw = pd.read_excel(file_path, sheet_name="IS-All", header=None)

df_raw = df_raw.dropna(axis=1, how='all')
df_raw.columns = range(df_raw.shape[1])

# ------------------------------------------------------------------
# 2. Extract and clean the Dates (same as before)
# ------------------------------------------------------------------
date_mask = df_raw[0].astype(str).str.startswith('Date', na=False)
df_raw['Date'] = np.where(date_mask, df_raw[0].astype(str), np.nan)
df_raw['Date'] = df_raw['Date'].ffill()
df_raw['Date'] = df_raw['Date'].astype(str).str.replace('Date :', '').str.strip()
df_raw['Date'] = df_raw['Date'].str.replace(r'(\d+)(st|nd|rd|th)', r'\1', regex=True)
df_raw['Date'] = pd.to_datetime(df_raw['Date'], format='mixed', errors='coerce')

# ------------------------------------------------------------------
# 3. NEW SAFETY CHECK
#    Find every 'GL Code' header row. A REAL month-block header always
#    has a 'Date :' label a few rows above it. If a 'GL Code' header
#    has NO Date label nearby, it's a stray/leftover table (like the
#    utility notes table we found after July) -> we skip it.
# ------------------------------------------------------------------
header_rows = df_raw.index[df_raw[0] == 'GL Code'].tolist()
date_rows = df_raw.index[date_mask].tolist()

valid_header_rows = []
skipped_header_rows = []
for h in header_rows:
    has_date_nearby = any(0 <= h - d <= 4 for d in date_rows)
    (valid_header_rows if has_date_nearby else skipped_header_rows).append(h)

if skipped_header_rows:
    print(f"⚠️  Skipped {len(skipped_header_rows)} extra table(s) at row(s) {skipped_header_rows}.")
    print("    These have GL Code headers but NO 'Date :' label above them -> ")
    print("    treated as leftover notes, not real monthly data.")

if not valid_header_rows:
    raise ValueError("No valid month blocks found - check the file format!")

# ------------------------------------------------------------------
# 4. Apply the correct headers (from the first valid block)
# ------------------------------------------------------------------
headers = df_raw.iloc[valid_header_rows[0]].tolist()
headers[-1] = 'Date'
df_raw.columns = headers

# ------------------------------------------------------------------
# 5. Filter: only numeric GL Code rows, AND only rows inside a
#    VALID block (this is what blocks the duplicate rows)
# ------------------------------------------------------------------
is_numeric_gl = pd.to_numeric(df_raw['GL Code'], errors='coerce').notna()

all_cutpoints = sorted(header_rows) + [len(df_raw)]
valid_row_mask = pd.Series(False, index=df_raw.index)
for h in valid_header_rows:
    end = all_cutpoints[all_cutpoints.index(h) + 1]
    valid_row_mask.loc[h:end - 1] = True

df_clean = df_raw[is_numeric_gl & valid_row_mask].copy()

# ------------------------------------------------------------------
# 6. Melt / Unpivot the Data (same as before)
# ------------------------------------------------------------------
id_cols = ['Date', 'GL Code', 'Description']
branch_cols = [c for c in df_clean.columns if c not in id_cols and pd.notna(c) and 'Total' not in str(c)]
df_melted = pd.melt(df_clean, id_vars=id_cols, value_vars=branch_cols, var_name='Branch', value_name='Amount')
df_melted = df_melted.dropna(subset=['Amount'])

# ------------------------------------------------------------------
# 7. Add Category and Signed Amount (same as before)
# ------------------------------------------------------------------
df_melted['Category'] = np.where(df_melted['GL Code'].astype(str).str.startswith('5'), 'Income', 'Expense')
df_melted['Signed Amount'] = np.where(df_melted['Category'] == 'Expense', -df_melted['Amount'], df_melted['Amount'])

# ------------------------------------------------------------------
# 8. Save
# ------------------------------------------------------------------
df_melted.to_excel("IS_Master_All.xlsx", index=False)
print(f"Rows in final table: {len(df_melted)}")
print("Data is 100% production-ready!")

⚠️  Skipped 1 extra table(s) at row(s) [578].
    These have GL Code headers but NO 'Date :' label above them -> 
    treated as leftover notes, not real monthly data.
Rows in final table: 2786
Data is 100% production-ready!
